
# Completed EV Charging Infrastructure Planning Models

## Models included

1. **Base MCLP** — maximize covered demand under a budget.
2. **Equity MCLP** — maximize equity-weighted covered demand and enforce minimum high-TDI coverage.
3. **Base FCLP** — minimize fixed station, charger, user travel, and grid connection costs while serving a required demand fraction.
4. **Equity FCLP** — FCLP with high-TDI minimum service constraints.




In [1]:

# Import the completed model library.
# Keep completed_ev_charging_models.py in the same folder as this notebook and the data files.

from completed_ev_charging_models import (
    EVConfig,
    load_ev_data,
    summarize_data,
    solve_base_mclp,
    solve_equity_mclp,
    solve_base_fclp,
    solve_equity_fclp,
    extract_solution,
)

import pandas as pd



## 1. Load and validate data

Expected files:

- `TrialDataNCDOT.xlsx`
- `Demand Calculation.xlsx`
- `Data.xlsx`
- `distance_matrice.xlsx`
- `power_substation.xlsx`
- `Max capacity of power_substations.xlsx`
- `raleigh_locations.csv`
- `nodes_locations.csv`

The model uses the filtered 74 candidate/demand locations and 37 Raleigh-area substations. `nodes_locations.csv` is retained as the broader raw node source/provenance file.


In [2]:

data = load_ev_data(".")
summary = summarize_data(data)
summary


,metric,value
0,candidate_sites,74.000000
1,demand_zones,74.000000
2,substations,37.000000
3,total_demand,2220.000000
4,min_demand,12.453816
5,max_demand,292.284303
6,min_tdi,6.000000
7,max_tdi,14.500000
8,max_travel_miles,13.766726
9,max_transmission_miles,29.898997



## 2. Planning assumptions


- `service_radius_miles=3.0` is much more meaningful than the old 15-mile threshold because maximum candidate-to-demand distance is only about 13.77 miles.
- `grid_available_fraction=0.02` uses only 2% of the listed substation final capacity for EV expansion. 
- `demand_units_per_port` translates demand score into charger-port capacity. 


In [3]:

cfg = EVConfig(
    service_radius_miles=3.0,
    max_grid_connection_miles=10.0,
    budget=800_000,
    fixed_station_cost=20_000,
    charger_port_cost=18_000,
    grid_connection_cost_per_mile=10_000,
    user_travel_cost_per_mile=1.0,
    demand_units_per_port=35.0,
    min_ports_per_open_station=2,
    max_ports_per_open_station=12,
    charger_power_mw=0.150,
    grid_available_fraction=0.02,
    required_demand_fraction=0.75,
    high_tdi_quantile=0.75,
    min_high_tdi_coverage_fraction=0.70,
    equity_weight_strength=1.0,
    mip_gap=0.01,
    time_limit_seconds=300,
)
cfg


EVConfig(service_radius_miles=3.0, max_grid_connection_miles=10.0, budget=800000, fixed_station_cost=20000, charger_port_cost=18000, grid_connection_cost_per_mile=10000, user_travel_cost_per_mile=1.0, demand_units_per_port=35.0, min_ports_per_open_station=2, max_ports_per_open_station=12, charger_power_mw=0.15, grid_available_fraction=0.02, required_demand_fraction=0.75, high_tdi_quantile=0.75, min_high_tdi_coverage_fraction=0.7, equity_weight_strength=1.0, mip_gap=0.01, time_limit_seconds=300)


## 3. Base MCLP



In [4]:

#Uncomment to solve after Gurobi is installed/licensed.

model, vars_dict = solve_base_mclp(data, cfg)
sol = extract_solution(data, model, vars_dict, cfg)
sol["open_stations"].head()


Set parameter Username
Set parameter LicenseID to value 2755839
Academic license - for non-commercial use only - expires 2026-12-16
Set parameter MIPGap to value 0.01
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i5-12450H, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.01

Optimize a model with 19648 rows, 11174 columns and 52762 nonzeros (Max)
Model fingerprint: 0x07d9223a
Model has 74 linear objective coefficients
Variable types: 2738 continuous, 8436 integer (8362 binary)
Coefficient statistics:
  Matrix range     [1e-01, 3e+05]
  Objective range  [1e+01, 3e+02]
  Bounds range     [1e+00, 1e+01]
  RHS range        [1e+00, 8e+05]
Found heuristic solution: objective -0.0000000
Presolve removed 17357 rows and 8563 columns
Presolve time: 0.06s
Presolved: 2291 rows

,station_id,latitude,longitude,ports,assigned_demand_units,assigned_zones,connected_substation,substation_latitude,substation_longitude,grid_distance_miles,power_mw
0,12,35.774549,-78.631485,12,419.987656,19,0,35.769038,-78.632319,0.498719,1.8
1,22,35.808006,-78.689632,11,384.545824,4,6,35.807958,-78.692485,0.207868,1.8
2,24,35.808131,-78.672028,9,314.957471,12,6,35.807958,-78.692485,1.490242,1.8
3,62,35.736782,-78.581090,6,209.896924,5,27,35.730063,-78.569641,1.030059,1.8



## 4. Equity MCLP



In [5]:

# Uncomment to solve.

model, vars_dict = solve_equity_mclp(data, cfg)
sol = extract_solution(data, model, vars_dict, cfg)
sol["open_stations"].head()


Set parameter MIPGap to value 0.01
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i5-12450H, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.01

Optimize a model with 19649 rows, 11174 columns and 52784 nonzeros (Max)
Model fingerprint: 0x506b5699
Model has 74 linear objective coefficients
Variable types: 2738 continuous, 8436 integer (8362 binary)
Coefficient statistics:
  Matrix range     [1e-01, 3e+05]
  Objective range  [2e+01, 3e+02]
  Bounds range     [1e+00, 1e+01]
  RHS range        [1e+00, 8e+05]
Presolve removed 17357 rows and 8555 columns
Presolve time: 0.05s
Presolved: 2292 rows, 2619 columns, 9332 nonzeros
Variable types: 490 continuous, 2129 integer (2055 binary)

Root relaxation: objective 1.919194e+03, 4057 iterations, 0.08 seconds (0.14 work units)


,station_id,latitude,longitude,ports,assigned_demand_units,assigned_zones,connected_substation,substation_latitude,substation_longitude,grid_distance_miles,power_mw
0,16,35.768278,-78.641469,11,383.849295,18,0,35.769038,-78.632319,0.670328,1.80
1,39,35.781912,-78.591663,7,244.177783,10,1,35.764480,-78.598414,1.641210,1.80
2,53,35.832185,-78.593466,8,277.431564,11,35,35.828273,-78.561781,2.334000,1.35
3,61,35.719480,-78.593172,10,339.075541,9,27,35.730063,-78.569641,1.961602,1.80



## 5. Base FCLP



In [ ]:

# Uncomment to solve.

model, vars_dict = solve_base_fclp(data, cfg)
sol = extract_solution(data, model, vars_dict, cfg)
for name, df in sol.items():
     df.to_csv(f"{name}_base_fclp.csv", index=False)
sol["open_stations"].head()


Set parameter MIPGap to value 0.01
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i5-12450H, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.01

Optimize a model with 19648 rows, 11174 columns and 49950 nonzeros (Min)
Model fingerprint: 0x80486e38
Model has 8288 linear objective coefficients
Variable types: 2738 continuous, 8436 integer (8362 binary)
Coefficient statistics:
  Matrix range     [1e-01, 6e+02]
  Objective range  [3e+00, 3e+05]
  Bounds range     [1e+00, 1e+01]
  RHS range        [1e+00, 2e+03]
Presolve removed 17357 rows and 8550 columns
Presolve time: 0.06s
Presolved: 2291 rows, 2624 columns, 8677 nonzeros
Variable types: 490 continuous, 2134 integer (2060 binary)
Found heuristic solution: objective 1.071126e+07
Found heuristic solution: objective 767

,station_id,latitude,longitude,ports,assigned_demand_units,assigned_zones,connected_substation,substation_latitude,substation_longitude,grid_distance_miles,power_mw
0,6,35.791378,-78.630676,12,418.552810,18,0,35.769038,-78.632319,2.010175,1.8
1,16,35.768278,-78.641469,12,418.278660,19,0,35.769038,-78.632319,0.670328,1.8
2,22,35.808006,-78.689632,12,416.620246,6,6,35.807958,-78.692485,0.207868,1.8
3,42,35.727841,-78.604845,12,415.458297,11,27,35.730063,-78.569641,2.574682,1.8



## 6. Equity FCLP


In [ ]:

# Uncomment to solve.

model, vars_dict = solve_equity_fclp(data, cfg)
sol = extract_solution(data, model, vars_dict, cfg)
for name, df in sol.items():
     df.to_csv(f"{name}_equity_fclp.csv", index=False)
sol["open_stations"].head()


Set parameter MIPGap to value 0.01
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i5-12450H, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.01

Optimize a model with 19649 rows, 11174 columns and 49972 nonzeros (Min)
Model fingerprint: 0xc6e06344
Model has 8288 linear objective coefficients
Variable types: 2738 continuous, 8436 integer (8362 binary)
Coefficient statistics:
  Matrix range     [1e-01, 6e+02]
  Objective range  [3e+00, 3e+05]
  Bounds range     [1e+00, 1e+01]
  RHS range        [1e+00, 2e+03]
Presolve removed 17357 rows and 8550 columns
Presolve time: 0.06s
Presolved: 2292 rows, 2624 columns, 8699 nonzeros
Variable types: 490 continuous, 2134 integer (2060 binary)
Found heuristic solution: objective 1.071126e+07
Found heuristic solution: objective 767

,station_id,latitude,longitude,ports,assigned_demand_units,assigned_zones,connected_substation,substation_latitude,substation_longitude,grid_distance_miles,power_mw
0,9,35.788577,-78.616604,12,416.587944,18,0,35.769038,-78.632319,2.095586,1.8
1,15,35.768783,-78.646920,12,419.123944,19,0,35.769038,-78.632319,1.064353,1.8
2,22,35.808006,-78.689632,12,414.464047,6,6,35.807958,-78.692485,0.207868,1.8
3,42,35.727841,-78.604845,12,415.458297,11,27,35.730063,-78.569641,2.574682,1.8



## 7. Final model recommendation


1. **Base MCLP** as the coverage-maximization planning baseline.
2. **Base FCLP** as the cost-minimization infrastructure planning model.
3. **Equity FCLP** as the final realistic model.


